# 01 — Data Preparation

**From Clinical Case Reports to Knowledge Graphs**

This notebook is Step 0 of the activity plan: it downloads the source files
from the [MultiCaRe dataset on Zenodo](https://doi.org/10.5281/zenodo.10079369)
and loads them into a single DuckDB database. Later notebooks (corpus
exploration, manual annotation, knowledge graph construction, vector
representations) all connect to the database produced here — they do not
re-download or re-parse the source files.

Only the files actually used by the activity are downloaded (`cases.parquet`,
`metadata.parquet`, `data_dictionary.csv`). The dataset also ships PubMed
Central image archives and image-caption tables (several GB) that this
activity does not use, so they are skipped.

## 1. Setup

Directory layout expected by this notebook (created automatically if missing):

```
to-kg/
├── data/                 <- downloaded files + clinical_cases.duckdb (gitignored)
└── notebooks/
    └── 01_data_preparation.ipynb   <- this notebook
```

In [1]:
from pathlib import Path

import duckdb
import requests
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "clinical_cases.duckdb"

DATA_DIR

PosixPath('/home/jovyan/work/to-kg/data')

## 2. Download source files from Zenodo

We resolve the **concept DOI** (`10.5281/zenodo.10079369`) through Zenodo's
`versions/latest` endpoint, so this notebook always fetches the current
version of the dataset instead of a version hardcoded at the time this
notebook was written. Files already present with the correct size are not
re-downloaded, so re-running this cell is cheap.

In [2]:
ZENODO_CONCEPT_ID = "10079369"  # concept DOI 10.5281/zenodo.10079369
FILES_TO_DOWNLOAD = ["cases.parquet", "metadata.parquet", "data_dictionary.csv"]


def get_latest_record(concept_id: str) -> dict:
    url = f"https://zenodo.org/api/records/{concept_id}/versions/latest"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def download_file(url: str, destination: Path, expected_size: int) -> None:
    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"  {destination.name} already present ({expected_size:,} bytes), skipping")
        return
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(destination, "wb") as fh, tqdm(
            total=expected_size, unit="B", unit_scale=True, desc=destination.name
        ) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                fh.write(chunk)
                bar.update(len(chunk))

In [3]:
record = get_latest_record(ZENODO_CONCEPT_ID)
print(
    f"Latest version: {record['metadata']['version']} "
    f"(record id {record['id']}, published {record['metadata']['publication_date']})"
)

files_by_key = {f["key"]: f for f in record["files"]}

for key in FILES_TO_DOWNLOAD:
    file_info = files_by_key[key]
    download_file(file_info["links"]["self"], DATA_DIR / key, file_info["size"])

Latest version: 3.0.1 (record id 20416562, published 2026-05-27)
  cases.parquet already present (168,061,894 bytes), skipping
  metadata.parquet already present (20,357,281 bytes), skipping
  data_dictionary.csv already present (6,291 bytes), skipping


## 3. Build the DuckDB database

Building an explicit DuckDB database file is not required for the
information-extraction steps later in the activity — DuckDB can query
Parquet files directly. We build it anyway because it gives students a
single, fast, queryable artifact (`data/clinical_cases.duckdb`) to explore
the corpus before any NLP work starts.

**Note on the raw file layout:** both source files are more nested than
`data_dictionary.csv` documents. `cases.parquet` has one row per *article*,
with a `cases` column holding a list of structs (one struct per patient
case: `age`, `case_id`, `case_text`, `gender`). `metadata.parquet` has one
row per article too, with every documented field (`title`, `authors`,
`journal`, `year`, `doi`, `pmid`, `mesh_terms`, `case_amount`, …) packed
into a single `article_metadata` struct column. The `CREATE TABLE`
statements below unnest/flatten both so the resulting `cases` and
`metadata` tables match the flat, one-row-per-case / one-row-per-article
columns the data dictionary describes.

In [4]:
con = duckdb.connect(str(DB_PATH))

con.execute(f"""
    CREATE OR REPLACE TABLE cases AS
    SELECT article_id, case_entry.*
    FROM (
        SELECT article_id, UNNEST(cases) AS case_entry
        FROM read_parquet('{(DATA_DIR / "cases.parquet").as_posix()}')
    )
""")

con.execute(f"""
    CREATE OR REPLACE TABLE metadata AS
    SELECT article_id, article_metadata.*
    FROM read_parquet('{(DATA_DIR / "metadata.parquet").as_posix()}')
""")

con.execute(f"""
    CREATE OR REPLACE TABLE data_dictionary AS
    SELECT * FROM read_csv_auto('{(DATA_DIR / "data_dictionary.csv").as_posix()}')
""")

con.sql("SHOW TABLES")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ cases           │
│ data_dictionary │
│ metadata        │
└─────────────────┘

In [5]:
con.sql("""
    SELECT 'cases' AS table_name, COUNT(*) AS n_rows FROM cases
    UNION ALL
    SELECT 'metadata', COUNT(*) FROM metadata
    UNION ALL
    SELECT 'data_dictionary', COUNT(*) FROM data_dictionary
""")

┌─────────────────┬────────┐
│   table_name    │ n_rows │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ cases           │  98641 │
│ metadata        │  76137 │
│ data_dictionary │     45 │
└─────────────────┴────────┘

## 4. Example queries — exploring the tables

These mirror the exploratory queries suggested in Step 1 of the activity
plan, now run against the DuckDB database instead of the raw Parquet file.

### Table schema

In [6]:
con.sql("DESCRIBE cases")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ case_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ case_text   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gender      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [7]:
con.sql("SELECT count(*) FROM cases")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        98641 │
└──────────────┘

### A random sample of cases

In [8]:
con.sql("SELECT * FROM cases USING SAMPLE 5 ROWS")

┌────────────┬────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### Text length distribution

In [9]:
con.sql("""
    SELECT
        MIN(LENGTH(case_text)) AS min_chars,
        MEDIAN(LENGTH(case_text)) AS median_chars,
        AVG(LENGTH(case_text))::INT AS avg_chars,
        MAX(LENGTH(case_text)) AS max_chars
    FROM cases
""")

┌───────────┬──────────────┬───────────┬───────────┐
│ min_chars │ median_chars │ avg_chars │ max_chars │
│   int64   │    double    │   int32   │   int64   │
├───────────┼──────────────┼───────────┼───────────┤
│         9 │       2543.0 │      2965 │     79243 │
└───────────┴──────────────┴───────────┴───────────┘

### Patient demographics

In [10]:
con.sql("""
    SELECT gender, COUNT(*) AS n_cases
    FROM cases
    GROUP BY gender
    ORDER BY n_cases DESC
""")

┌─────────────┬─────────┐
│   gender    │ n_cases │
│   varchar   │  int64  │
├─────────────┼─────────┤
│ Male        │   47069 │
│ Female      │   44778 │
│ Unknown     │    6679 │
│ Transgender │     115 │
└─────────────┴─────────┘

In [11]:
con.sql("""
    SELECT
        MIN(age) AS min_age,
        MEDIAN(age) AS median_age,
        AVG(age)::INT AS avg_age,
        MAX(age) AS max_age
    FROM cases
    WHERE age IS NOT NULL
""")

┌─────────┬────────────┬─────────┬─────────┐
│ min_age │ median_age │ avg_age │ max_age │
│ double  │   double   │  int32  │ double  │
├─────────┼────────────┼─────────┼─────────┤
│     0.0 │       42.0 │      41 │   120.0 │
└─────────┴────────────┴─────────┴─────────┘

### Joining `cases` with `metadata`

`cases` and `metadata` share the `article_id` (PMCID) column, so we can
bring in publication year, journal, license, etc.

In [12]:
con.sql("""
    SELECT m.year, COUNT(*) AS n_cases
    FROM cases AS c
    JOIN metadata AS m USING (article_id)
    GROUP BY m.year
    ORDER BY m.year
""")

┌─────────┬─────────┐
│  year   │ n_cases │
│ varchar │  int64  │
├─────────┼─────────┤
│ 1990    │      11 │
│ 1991    │       7 │
│ 1992    │       5 │
│ 1993    │      10 │
│ 1994    │       9 │
│ 1995    │       9 │
│ 1996    │      10 │
│ 1997    │      16 │
│ 1998    │      13 │
│ 1999    │      14 │
│  ·      │       · │
│  ·      │       · │
│  ·      │       · │
│ 2017    │    5905 │
│ 2018    │    6734 │
│ 2019    │    7359 │
│ 2020    │    7562 │
│ 2021    │    8120 │
│ 2022    │    8715 │
│ 2023    │   11616 │
│ 2024    │    8893 │
│ 2025    │    6513 │
│ 2026    │    2250 │
└─────────┴─────────┘
       37 rows     
     (20 shown)     

In [13]:
con.sql("""
    SELECT journal, COUNT(DISTINCT article_id) AS n_articles
    FROM metadata
    GROUP BY journal
    ORDER BY n_articles DESC
    LIMIT 10
""")

┌────────────────────────┬────────────┐
│        journal         │ n_articles │
│        varchar         │   int64    │
├────────────────────────┼────────────┤
│ Front Oncol            │       2434 │
│ Cureus                 │       2413 │
│ Surg Neurol Int        │       2359 │
│ SAGE Open Med Case Rep │       1983 │
│ Pan Afr Med J          │       1820 │
│ Case Rep Med           │       1726 │
│ Front Pediatr          │       1434 │
│ Front Med (Lausanne)   │       1404 │
│ Int J Surg Case Rep    │       1375 │
│ Front Neurol           │       1269 │
└────────────────────────┴────────────┘
  10 rows                   2 columns

In [14]:
con.sql("""
    SELECT license, COUNT(*) AS n_articles
    FROM metadata
    GROUP BY license
    ORDER BY n_articles DESC
""")

┌─────────────┬────────────┐
│   license   │ n_articles │
│   varchar   │   int64    │
├─────────────┼────────────┤
│ CC BY       │      49838 │
│ CC BY-NC    │      16648 │
│ CC BY-NC-SA │       9598 │
│ CC0         │         53 │
└─────────────┴────────────┘

## 5. Confronting the extracted metadata with the source paper

The dataset description paper (Nievas Offidani et al., *An Open-Source
Clinical Case Dataset for Medical Image Classification and Multimodal AI
Applications*, *Data*, 2026, [doi:10.3390/data10080123](https://doi.org/10.3390/data10080123))
reports approximately **93,816 clinical cases** in the abstract. Because this
notebook always downloads the *latest* Zenodo version (Section 2), while the
paper describes a snapshot taken at publication time, some drift between the
two counts is expected — the query below quantifies it rather than assuming
the numbers must match.

In [15]:
PAPER_REPORTED_CASES = 93_816  # abstract of Nievas Offidani et al., Data (2026)

actual_cases = con.sql("SELECT COUNT(*) FROM cases").fetchone()[0]
diff = actual_cases - PAPER_REPORTED_CASES

print(f"Cases in the downloaded dataset version {record['metadata']['version']}: {actual_cases:,}")
print(f"Cases reported in the paper:                                    {PAPER_REPORTED_CASES:,}")
print(f"Difference:                                                     {diff:+,} ({diff / PAPER_REPORTED_CASES:+.1%})")

Cases in the downloaded dataset version 3.0.1: 98,641
Cases reported in the paper:                                    93,816
Difference:                                                     +4,825 (+5.1%)


## 6. Wrap up

The database now lives at `data/clinical_cases.duckdb` with three tables:
`cases`, `metadata`, and `data_dictionary`. We close the connection here so
the file is not left locked; later notebooks reopen it with:

```python
import duckdb
con = duckdb.connect("../data/clinical_cases.duckdb")
```

In [16]:
con.close()